<img src="https://www.fciencias.unam.mx/sites/default/files/logoFC_2.png" width="500" align="left"/>
<p align="right">
<b>Lingüística Computacional</b>
<br><b>Práctica 5: Fine-tunning y puesta en producción de modelos</b>
<br><b>Profesora:</b> Dra. Ximena Gutiérrez Vasques
<br><b>Ayudante Teo:</b> Ximena de la Luz Contreras Mendoza
<br><b>Ayudante Lab:</b> Diego Alberto Barriga Martínez
<br><b>Alumna:</b> Ortega Hernández Zaira Daniela
<br><b>Mayo, 2026</b>
</p>

##**Práctica 5: Fine-tunning y puesta en producción de modelos**


In [1]:
!pip install transformers datasets evaluate accelerate codecarbon gradio

In [2]:
from datasets import load_dataset
from transformers import AutoTokenizer

# Cargamos el dataset (versión simplificada de 28 etiquetas)
dataset = load_dataset("go_emotions", "simplified")

# Usaremos DistilBERT por ser pequeño y eficiente
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

# Tokenizamos el dataset
tokenized_dataset = dataset.map(preprocess_function, batched=True)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
import numpy as np
import evaluate

# 1. Cargar el dataset
dataset = load_dataset("go_emotions", "simplified")

# 2. Sobrescribimos la columna 'labels' original para que deje de ser una lista
dataset = dataset.map(lambda x: {"labels": x["labels"][0]})

# 3. Tokenización
model_checkpoint = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    return tokenizer(examples["text"], truncation=True, padding="max_length", max_length=128)

tokenized_dataset = dataset.map(preprocess_function, batched=True)

# 4. Configurar modelo
# Nota: Como sobrescribimos 'labels', para obtener los nombres originales cargamos la info directo del dataset puro
num_etiquetas = 28 # GoEmotions simplificado tiene 28 emociones
model = AutoModelForSequenceClassification.from_pretrained(
    model_checkpoint,
    num_labels=num_etiquetas
)

# 5. Métricas
metric = evaluate.load("accuracy")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# 6. Entrenamiento
training_args = TrainingArguments(
    output_dir="./result_emotions",
    eval_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=10,
    weight_decay=0.01,
    logging_steps=10,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"].shuffle(seed=42).select(range(2000)),
    eval_dataset=tokenized_dataset["validation"].select(range(500)),
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
)

trainer.train()

Map:   0%|          | 0/5426 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy
1,2.679989,2.599234,0.318000
2,2.203462,2.247885,0.380000
3,1.892236,2.067300,0.422000
4,1.519488,1.977664,0.450000
5,1.486903,1.952695,0.448000
6,1.167925,1.953271,0.454000
7,0.935500,1.959976,0.474000
8,0.825072,1.955306,0.476000
9,0.909130,1.960768,0.464000
10,0.659225,1.964571,0.468000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=1250, training_loss=1.5105947914123534, metrics={'train_runtime': 274.2683, 'train_samples_per_second': 72.921, 'train_steps_per_second': 4.558, 'total_flos': 662644101120000.0, 'train_loss': 1.5105947914123534, 'epoch': 10.0})

In [4]:
import gradio as gr
from transformers import pipeline
import numpy as np
from codecarbon import OfflineEmissionsTracker

# 1. Cargamos el modelo y el tokenizador que guarda Colab
path_modelo = "./mi_modelo_emociones"
pipe = pipeline("text-classification", model=path_modelo, tokenizer=path_modelo)

# Diccionario mapeando los IDs numéricos a los nombres reales de las emociones de GoEmotions
id2label = {
    0: "admiration", 1: "amusement", 2: "anger", 3: "annoyance", 4: "approval",
    5: "caring", 6: "confusion", 7: "curiosity", 8: "desire", 9: "disappointment",
    10: "disapproval", 11: "disgust", 12: "embarrassment", 13: "excitement",
    14: "fear", 15: "gratitude", 16: "grief", 17: "joy", 18: "love",
    19: "nervousness", 20: "optimism", 21: "pragmatic", 22: "realization",
    23: "relief", 24: "remorse", 25: "sadness", 26: "surprise", 27: "neutral"
}

# 2. Definimos la función de predicción
def predict_emotion(text):
    if not text.strip():
        return {"Por favor, escribe algo": 1.0}

    # --- INICIAMOS CODECARBON MANUALMENTE ---
    # Usamos log_level="error" para que no llene la consola de Colab con tantos mensajes
    tracker = OfflineEmissionsTracker(project_name="Prediccion_Emociones", country_iso_code="USA", log_level="error")
    tracker.start()

    # El modelo hace la predicción
    results = pipe(text)

    # --- DETENEMOS CODECARBON Y GUARDAMOS EL VALOR ---
    emisiones = tracker.stop()

    # Formateamos el resultado de CO2 para que se vea bonito en pantalla
    texto_emisiones = f"{emisiones:.8f} kg de CO2eq"

    # Procesamos las emociones
    output = {}
    for res in results:
        label_raw = res["label"]
        score = res["score"]

        # Extraemos el número del label para buscar su nombre real
        try:
            label_id = int(label_raw.split("_")[-1])
            label_name = id2label.get(label_id, label_raw)
        except:
            label_name = label_raw

        output[label_name] = float(score)

    return output, texto_emisiones

# 3. Construimos la interfaz de Gradio
demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Escribe un comentario aquí (en inglés)...",
        label="Texto de entrada"
    ),
    outputs=[
        gr.Label(num_top_classes=3, label="Emociones Detectadas"),
        gr.Textbox(label="Huella de Carbono de esta predicción")
    ],
    title="Análisis de Emociones - Práctica 5",
    description="Prototipo que identifica emociones usando DistilBERT. Calcula y muestra en tiempo real las emisiones de CO2 generadas por la inferencia.",
    examples=[
        ["I am so proud of your achievements! Congratulations!"],
        ["What? I don't understand what is happening here."],
        ["I'm feeling a bit down today, everything seems gray."]
    ]
)

# Esto arranca la app localmente en Colab para probarla antes de subirla
if __name__ == "__main__":
    demo.launch(debug=True)

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://439838f87db6d68a1e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


[codecarbon INFO @ 05:26:53] offline tracker init
[codecarbon WARNING @ 05:26:53] Multiple instances of codecarbon are allowed to run at the same time.


Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://439838f87db6d68a1e.gradio.live


A continuación se anexan los pasos para descargar los archivos necesarios para subirlo a Hugging Face

In [5]:
# Guardar la app para subir a Hugging Face
%%writefile app.py
import gradio as gr
from transformers import pipeline
import numpy as np
from codecarbon import OfflineEmissionsTracker

# 1. Cargamos el modelo y el tokenizador que guarda Colab
path_modelo = "./mi_modelo_emociones"
pipe = pipeline("text-classification", model=path_modelo, tokenizer=path_modelo)

# Diccionario mapeando los IDs numéricos a los nombres reales de las emociones de GoEmotions
id2label = {
    0: "admiration", 1: "amusement", 2: "anger", 3: "annoyance", 4: "approval",
    5: "caring", 6: "confusion", 7: "curiosity", 8: "desire", 9: "disappointment",
    10: "disapproval", 11: "disgust", 12: "embarrassment", 13: "excitement",
    14: "fear", 15: "gratitude", 16: "grief", 17: "joy", 18: "love",
    19: "nervousness", 20: "optimism", 21: "pragmatic", 22: "realization",
    23: "relief", 24: "remorse", 25: "sadness", 26: "surprise", 27: "neutral"
}

# 2. Definimos la función de predicción
def predict_emotion(text):
    if not text.strip():
        return {"Por favor, escribe algo": 1.0}

    # --- INICIAMOS CODECARBON MANUALMENTE ---
    # Usamos log_level="error" para que no llene la consola de Colab con tantos mensajes
    tracker = OfflineEmissionsTracker(project_name="Prediccion_Emociones", country_iso_code="USA", log_level="error")
    tracker.start()

    # El modelo hace la predicción
    results = pipe(text)

    # --- DETENEMOS CODECARBON Y GUARDAMOS EL VALOR ---
    emisiones = tracker.stop()

    # Formateamos el resultado de CO2 para que se vea bonito en pantalla
    texto_emisiones = f"{emisiones:.8f} kg de CO2eq"

    # Procesamos las emociones
    output = {}
    for res in results:
        label_raw = res["label"]
        score = res["score"]

        # Extraemos el número del label para buscar su nombre real
        try:
            label_id = int(label_raw.split("_")[-1])
            label_name = id2label.get(label_id, label_raw)
        except:
            label_name = label_raw

        output[label_name] = float(score)

    return output, texto_emisiones

# 3. Construimos la interfaz de Gradio
demo = gr.Interface(
    fn=predict_emotion,
    inputs=gr.Textbox(
        lines=3,
        placeholder="Escribe un comentario aquí (en inglés)...",
        label="Texto de entrada"
    ),
    outputs=[
        gr.Label(num_top_classes=3, label="Emociones Detectadas"),
        gr.Textbox(label="Huella de Carbono de esta predicción")
    ],
    title="Análisis de Emociones - Práctica 5",
    description="Prototipo que identifica emociones usando DistilBERT. Calcula y muestra en tiempo real las emisiones de CO2 generadas por la inferencia.",
    examples=[
        ["I am so proud of your achievements! Congratulations!"],
        ["What? I don't understand what is happening here."],
        ["I'm feeling a bit down today, everything seems gray."]
    ]
)

# Esto arranca la app localmente en Colab para probarla antes de subirla
if __name__ == "__main__":
    demo.launch(debug=True)

Overwriting app.py


In [6]:
# Guardamos el modelo ya entrenado y su tokenizador en la carpeta final
trainer.save_model("./mi_modelo_emociones")
tokenizer.save_pretrained("./mi_modelo_emociones")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('./mi_modelo_emociones/tokenizer_config.json',
 './mi_modelo_emociones/tokenizer.json')

In [7]:
# Comprimimos la carpeta del modelo
!zip -r modelo.zip ./mi_modelo_emociones

updating: mi_modelo_emociones/ (stored 0%)
updating: mi_modelo_emociones/tokenizer_config.json (deflated 42%)
updating: mi_modelo_emociones/model.safetensors (deflated 8%)
updating: mi_modelo_emociones/training_args.bin (deflated 53%)
updating: mi_modelo_emociones/config.json (deflated 66%)
updating: mi_modelo_emociones/tokenizer.json (deflated 71%)
